# 07 · Frontend Streamlit

**Propósito.** Documentar y verificar `src/frontend/app.py`: un formulario de 14 días que llama a la API por HTTP (`POST /predict`) y muestra la predicción, su interpretación y la versión del modelo, con mensajes comprensibles ante fallas.

| Elemento | Implementación |
|---|---|
| Configuración | `API_URL` desde `.env` (por defecto `http://127.0.0.1:8000`), editable en la barra lateral |
| Entrada | 14 `number_input` (`t-14` … `t-1`) con valores de ejemplo del canal |
| Salida | Vistas estimadas, versión y familia del modelo, nombre en Unity Catalog, interpretación y JSON de solicitud/respuesta |
| Errores | Conexión rechazada → “Error de conexión…”; HTTP 422 → “Error de validación en la API” con el detalle |

In [1]:
import os, sys, json
from pathlib import Path

# Los notebooks se ejecutan desde la raíz del repositorio para compartir rutas con src/
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

## 1. Arranque de los procesos

```bash
uv run uvicorn src.api.main:app --host 127.0.0.1 --port 8000   # terminal 1
uv run streamlit run src/frontend/app.py                        # terminal 2 -> http://localhost:8501
```

Las pruebas siguientes usan `streamlit.testing.v1.AppTest`, que ejecuta el script real del frontend sin navegador; las solicitudes HTTP sí van a la API real, que debe estar en ejecución.

In [2]:
from streamlit.testing.v1 import AppTest
import requests

API_URL = os.getenv("API_URL", "http://127.0.0.1:8000")
try:
    api_viva = requests.get(f"{API_URL}/health", timeout=5).ok
except requests.exceptions.ConnectionError:
    api_viva = False
print("API en ejecución:", api_viva, "|", API_URL)

def enviar_formulario(api_url=None, cambios=None):
    at = AppTest.from_file(str(RAIZ / "src" / "frontend" / "app.py"), default_timeout=60).run()
    if api_url:
        at.text_input[0].set_value(api_url).run()
    for etiqueta, valor in (cambios or {}).items():
        next(n for n in at.number_input if n.label == etiqueta).set_value(valor)
    boton = next(b for b in at.button if b.label == "Obtener Predicción")
    boton.click().run()
    return at

def lags_en_orden(at):
    # Streamlit enumera los widgets por columna; se ordenan por etiqueta (t-14 ... t-1)
    return [n.value for n in sorted(at.number_input, key=lambda n: -int(n.label.split("t-")[1]))]

API en ejecución: True | http://127.0.0.1:8000


## 2. Caso integrado: formulario → API → predicción

In [3]:
at = enviar_formulario()
lags = lags_en_orden(at)
print("Valores enviados (t-14 … t-1):", lags)
print("Mensajes:", [s.value for s in at.success] or [e.value for e in at.error])
for m in at.metric:
    print(f"  {m.label}: {m.value}")
for i in at.info:
    print("Interpretación:", i.value)

if api_viva:
    directo = requests.post(f"{API_URL}/predict", json={"lags_14_dias": lags}, timeout=10).json()
    assert at.metric[0].value == directo["predicted_views_formatted"]
    print("La predicción del frontend coincide con la respuesta de la API:", directo["predicted_views_formatted"],
          "| versión", directo["model_version"])

2026-10-26 19:40:42.313 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Valores enviados (t-14 … t-1): [3529320.0, 3816846.0, 1660823.0, 3210395.0, 3791115.0, 2218409.0, 1034500.0, 3077739.0, 3434652.0, 838785.0, 1615544.0, 1548594.0, 2100450.0, 1950300.0]
Mensajes: ['Predicción calculada correctamente']
  Vistas Estimadas (Día t): 4,599,233
  Versión del Modelo: v2 (random-forest)
  Modelo Registrado: workspace.default.pcd_otono_2026_tubecast
Interpretación: Se estima que los videos publicados el día t acumularán 4,599,233 vistas en total, un 90.3% por encima del promedio de los últimos 14 días (2,416,248).
La predicción del frontend coincide con la respuesta de la API: 4,599,233 | versión 2


## 3. Manejo de errores

In [4]:
# a) Validación: un valor negativo pasa el formulario pero la API lo rechaza (422)
at = enviar_formulario(cambios={"Día t-1": -5.0})
print("Errores:", [e.value for e in at.error])
print("Detalle mostrado:", [j.value for j in at.json][:1])
print("Métricas mostradas:", len(at.metric), "(no se muestra una predicción)")

2026-10-26 19:40:43.715 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Errores: ['Error de validación en la API (HTTP 422)']
Detalle mostrado: ['{"detail": [{"type": "value_error", "loc": ["body", "lags_14_dias"], "msg": "Value error, Las vistas no pueden ser negativas (valor recibido: -5.0 en posici\\u00f3n 13).", "input": [3529320.0, 3816846.0, 1660823.0, 3210395.0, 3791115.0, 2218409.0, 1034500.0, 3077739.0, 3434652.0, 838785.0, 1615544.0, 1548594.0, 2100450.0, -5.0], "ctx": {"error": {}}}]}']
Métricas mostradas: 0 (no se muestra una predicción)


In [5]:
# b) Conexión: la API configurada no responde
at = enviar_formulario(api_url="http://127.0.0.1:9")
print("Errores:", [e.value for e in at.error])
assert any("Error de conexión" in e.value for e in at.error)
print("Mensaje de conexión comprensible: OK")

2026-10-26 19:40:44.249 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Errores: ['Error de conexión: No fue posible comunicarse con la API en http://127.0.0.1:9. Verifica que el servidor Uvicorn esté en ejecución.']
Mensaje de conexión comprensible: OK


**Interpretación.** El frontend no carga el modelo ni replica el preprocesamiento: solo envía los 14 valores por HTTP y presenta lo que devuelve la API, incluida la versión exacta del Registry que respondió. Así, frontend y API están desacoplados (patrón *model-as-service* del mapa del sistema). Los errores de conexión y de validación se muestran como mensajes en español, sin valores inventados.

**Limitaciones.** El usuario debe capturar manualmente las vistas de los 14 días; un siguiente paso sería consultarlas automáticamente a partir del canal y la fecha.

Volver al [informe](../informe/informe-final.ipynb).